## Prerequisites

This notebook demonstrates how to invoke a **deployed real-time inference endpoint** on Azure for the Clinical De-identification model. Follow the steps below in order before running this notebook.

### Deployment Sequence

1. **Acquire the Azure Application Offer** — Subscribe to the *clinical_deidentification_docwise_wip_en* offer in the [Azure Marketplace](https://marketplace.azure.com) to obtain the Solution Template.
2. **Launch guided deployment** — From the Marketplace listing, start the guided deployment flow for the solution.
3. **Supply configuration** — Complete the guided UX forms (defined by the offer's `createUiDefinition`) with the required parameters.
4. **Deploy resources** — Confirm deployment; Azure provisions the resources defined in the offer's `mainTemplate.json`.
5. **Wait for deployment to succeed** — Verify the deployment reaches a *Succeeded* state before proceeding. Note the endpoint name from the deployment output.
6. **Run this notebook** — Execute the cells below against the real-time inference endpoint provisioned by the deployment.

### Environment

- **Notebook runtime**: This notebook is intended to run in **Azure Machine Learning workspace notebooks** (e.g. via an Azure ML Studio compute instance or Studio notebooks).
- **Jupyter kernel**: Use a kernel with the `azure.ai.ml` Python package installed, for example **Python 3.10 — SDK v2** (`python310-sdkv2`).
- **Authentication**: The kernel must be authenticated to Azure so that Azure ML SDK calls succeed (e.g. `DefaultAzureCredential()`). The authenticated identity must be granted Azure RBAC permissions appropriate for reading the workspace and online endpoint configuration, and for retrieving endpoint credentials. Consult your organization's access-policy guidance and the [Azure ML endpoint credential documentation](https://learn.microsoft.com/azure/machine-learning/how-to-manage-endpoints) for details.
- **Network access**: The notebook's compute instance must be able to reach the endpoint's `scoring_uri` (no outbound firewall or VNET restrictions that block it).


## Clinical Deidentification German

Deidentification is essential for safeguarding patient privacy in clinical data, including texts, PDFs, images, and DICOM files containing Protected Health Information (PHI). PHI encompasses various health-related data, including common identifiers such as name, address, birth date, and Social Security Number.

- **Model**: [clinical_deidentification_docwise_wip_de](https://nlp.johnsnowlabs.com/2024/11/04/clinical_deidentification_docwise_wip_de.html)
- **Model Description**: This pipeline can be used to deidentify PHI information from medical texts in German language. The PHI information will be masked and obfuscated in the resulting text.
The pipeline can mask and obfuscate: `LOCATION`, `DATE`, `NAME`, `ID`, `AGE`, `PROFESSION`, `CONTACT`, `ORGANIZATION`, `DOCTOR`, `CITY`, `COUNTRY`, `STREET`, `PATIENT`, `PHONE`, `HOSPITAL`, `STATE`, `DLN`, `SSN`, `ZIP`, `ACCOUNT`, `LICENSE`, `PLATE`, `VIN`, `MEDICALRECORD`, `EMAIL`, `URL` entities.

In [ ]:
ENDPOINT_NAME = "<CUSTOMER_TO_PUT_DEPLOYED_ENDPOINT_NAME>"

### Input Format

To use the model, you need to provide input in one of the following supported formats:

#### JSON Format

Provide input as JSON. We support two variations within this format:

1. **Array of Text Documents**: 
   Use an array containing multiple text documents. Each element represents a separate text document.

   ```json
   {
       "text": [
           "Text document 1",
           "Text document 2",
           ...
       ]
   }

    ```

2. **Single Text Document**:
   Provide a single text document as a string.


   ```json
    {
        "text": "Single text document"
    }
   ```

#### JSON Lines (JSONL) Format

Provide input in JSON Lines format, where each line is a JSON object representing a text document.

```
{"text": "Text document 1"}
{"text": "Text document 2"}
```

### Important Parameters

- **masking_policy**: `str`

    Users can select a masking policy to determine how sensitive entities are handled:

    Example: "**Dr. Hans-Wolfgang Weihmann - RM57, Städt Klinikum Dresden-Friedrichstadt, Friedrichstraße 41, Dresden**"

    - **masked**: Default policy that masks entities with their type.

      -> 'Dr.  `<PATIENT>` - `<USERNAME>`, `<HOSPITAL>`, `<STREET>`, `<CITY>`'

    - **obfuscated**: Replaces sensitive entities with random values of the same type.

      -> 'Dr.  `Karl-August Blümel` - `RP400`, `University Hospital Cologne`, `Fadime-Pölitz-Allee`, `Böblingen`'

    - **masked_fixed_length_chars**: Masks entities with a fixed length of asterisks (\*).

      -> 'Dr.  `****` - `****`, `****`, `****`, `****`'

    - **masked_with_chars**: Masks entities with asterisks (\*) matching the length of each entity.

      -> 'Dr.  [`*********************`] - [`****`], [`*************************************`], [`******************`], [`*******`]'
    
You can specify these parameters in the input as follows:

```json
{
    "text": [
        "Text document 1",
        "Text document 2",
        ...
    ],
    "masking_policy": "masked",
}
```

> **Note:** Demonstration inputs used with this endpoint should consist of approved synthetic or properly de-identified data. Never submit real protected health information (PHI) without explicit authorization and the required safeguarding controls.


### Authentication and Endpoint Lookup

The cell below uses `DefaultAzureCredential()` to authenticate to Azure, then retrieves the endpoint's scoring URI and primary access key from the workspace configuration. If you see authentication errors, verify that you are logged into Azure Cloud Shell / your compute instance and that your identity has sufficient role assignments on the workspace.


In [3]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential
import requests
import json


ml_client = MLClient.from_config(DefaultAzureCredential())
endpoint_url = ml_client.online_endpoints.get(ENDPOINT_NAME).scoring_uri
key = ml_client.online_endpoints.get_keys(ENDPOINT_NAME).primary_key


def invoke_realtime_endpoint(record, content_type="application/json", accept="application/json"):
    response = requests.post(
        endpoint_url,
        headers={
            "Authorization": f"Bearer {key}",
            "Content-Type": content_type,
        },
        data=json.dumps(record) if content_type == "application/json" else record,
     
    )

    response_body = response.content.decode("utf-8")

    if accept == "application/json":
        return json.loads(response_body)
    elif accept == "application/jsonlines":
        return [json.loads(line) for line in response_body.strip().split('\n') if line]
    else:
        raise ValueError(f"Unsupported accept type: {accept}")

Found the config file in: ./config.json


### Sample Data (Synthetic — for demonstration only)

The following cells define sample clinical text documents used in the examples below. When using this notebook with your own data, ensure all inputs are approved synthetic or properly de-identified. Never submit real protected health information (PHI) without explicit authorization and appropriate controls.


In [ ]:
docs = [
    """Dr. Hans-Wolfgang Weihmann - RM57, Städt Klinikum Dresden-Friedrichstadt, Friedrichstraße 41, Dresden""",

    """Er arbeitete bis 24.08.1940 - Gärtner bei Planten un Blomen in Hamburg, verbrannte sich an beiden Beinen - entwickelte Geschwüre. Der Patient konsultierte Dr. Klein im September.""",
    ]


sample_text = """Zusammenfassung : Michael Berger wird am Morgen des 12 Dezember 2018 ins St.Elisabeth Krankenhaus ei... 
Zusammenfassung : Michael Berger wird am Morgen des 12 Dezember 2018 ins St.Elisabeth Krankenhaus eingeliefert. Herr Michael Berger ist 76 Jahre alt und hat zu viel Wasser in den Beinen.Persönliche Daten :ID-Nummer: T0110053F Platte A-BC124 Kontonummer: DE89370400440532013000 SSN : 13110587M565 Lizenznummer: B072RRE2I55 Adresse : St.Johann-Straße 13 19300 """

### JSON

#### Example 1: masked (default-policy)

In [5]:
input_json_data = {"text": sample_text}
response_json = invoke_realtime_endpoint(input_json_data, content_type="application/json", accept="application/json")
response_json

{'predictions': 'Zusammenfassung : <PATIENT> wird am Morgen des <DATE> ins <HOSPITAL> ei... \nZusammenfassung : <PATIENT> wird am Morgen des <DATE> ins <HOSPITAL> eingeliefert. Herr <NAME> <NAME> ist <AGE> Jahre alt und hat zu viel Wasser in den Beinen.Persönliche Daten :ID-Nummer: <ID> Platte <PLATE> Kontonummer: <ACCOUNT> SSN : <SSN>Lizenznummer: <DLN> Adresse : <STREET> <ZIP>'}

#### Example 2: obfuscated

In [6]:
input_json_data = {"text": sample_text, "masking_policy": "obfuscated"}
response_json = invoke_realtime_endpoint(input_json_data, content_type="application/json", accept="application/json")
response_json

{'predictions': 'Zusammenfassung : Zdravko Sommer wird am Morgen des 10 Februar 2019 ins Klinikum Osnabrück ei... \nZusammenfassung : Zdravko Sommer wird am Morgen des 10 Februar 2019 ins Klinikum Osnabrück eingeliefert. Herr Zdravko Sommer ist 71 Jahre alt und hat zu viel Wasser in den Beinen.Persönliche Daten :ID-Nummer: G1881162W Platte L-IF859 Kontonummer: CZ74201911991625182111 SSN : 82881670B636Lizenznummer: I105MMZ5N66 Adresse : Scheuermannstr. 33 84211'}

### JSON Lines

In [7]:
def create_jsonl(records, masking_policy=None):
    json_records = []

    if isinstance(records, str):
        records = [records]

    for text in records:
        record = {"text": text}

        if masking_policy is not None:
            record["masking_policy"] = masking_policy
        json_records.append(record)

    json_lines = '\n'.join(json.dumps(record) for record in json_records)
    return json_lines


#### Example 1: masked (default-policy)

In [10]:
input_jsonl_data = create_jsonl(docs, masking_policy="masked")
data = invoke_realtime_endpoint(input_jsonl_data, content_type="application/jsonlines" , accept="application/jsonlines" )
data

[{'predictions': 'Dr. <NAME> Weihmann - <USERNAME>, <HOSPITAL>, <STREET>, <CITY>'},
 {'predictions': 'Er arbeitete bis <DATE> - <PROFESSION> bei <ORGANIZATION> in <STATE>, verbrannte sich an beiden Beinen - entwickelte Geschwüre. Der Patient konsultierte Dr. <NAME> im <NAME>.'}]

#### Example 2: obfuscated

In [11]:
input_jsonl_data = create_jsonl(docs, masking_policy="obfuscated")
data = invoke_realtime_endpoint(input_jsonl_data, content_type="application/jsonlines" , accept="application/jsonlines" )
data

[{'predictions': 'Dr. Meinhard Weihmann - FRED.EWALD, Sankt Maria Krankenhaus Dresden, Henkring 112, Bruchsal'},
 {'predictions': 'Er arbeitete bis 23.10.1940 - Laborant bei Deutsche Bank Ag in Bremen, verbrannte sich an beiden Beinen - entwickelte Geschwüre. Der Patient konsultierte Dr. Thorsten im Hassan.'}]